In [11]:
import numpy as np
import pandas as pd
from scipy.signal import find_peaks
from scipy.fft import rfft, rfftfreq
import antropy as ant
from nolds.measures import sampen
import glob
import os

In [5]:
# -----------------------------
# Robust MAD helper
# -----------------------------
def calcMAD(x):
    """Median Absolute Deviation"""
    med = np.median(x)
    return np.median(np.abs(x - med))


# -----------------------------
# Bandpower via FFT
# -----------------------------
def calcBandpower(signal, fs, fmin, fmax):
    N = len(signal)
    freqs = rfftfreq(N, d=1/fs)
    fft_vals = np.abs(rfft(signal))**2

    band_mask = (freqs >= fmin) & (freqs <= fmax)
    return np.sum(fft_vals[band_mask]), np.sum(fft_vals)


# -----------------------------
# Steering Feature Extraction
# -----------------------------
def calcSteeringFeatures(window_df,
                      steer_col="wheel_position",
                      time_col="timestamp",
                      lane_col=None):
    """
    Computes all steering behavior features described in your theory.
    Returns dict of features.
    """

    steer = window_df[steer_col].values
    t = window_df[time_col].values

    # Sampling frequency
    dt = np.median(np.diff(t))
    fs = 1 / dt

    features = {}

    # ============================================================
    # 1) Steering Jerk = 3rd derivative
    # ============================================================

    d1 = np.gradient(steer, dt)
    d2 = np.gradient(d1, dt)
    jerk = np.gradient(d2, dt)

    absjerk = np.abs(jerk)

    features["jerk_mean"] = np.mean(jerk)
    features["jerk_std"] = np.std(jerk)
    features["jerk_q95"] = np.quantile(absjerk, 0.95)

    # Extreme peaks: threshold = median + 6*MAD
    thr = np.median(absjerk) + 6 * calcMAD(absjerk)
    peaks, _ = find_peaks(absjerk, height=thr)

    features["jerk_extreme_peaks"] = len(peaks)

    # ============================================================
    # 2) Steering Entropy Measures
    # ============================================================

    r = 0.2 * np.std(steer)

    # Sample Entropy
    features["entropy_sample"] = sampen(steer)

    # Approximate Entropy
    features["entropy_approx"] = ant.app_entropy(steer)

    # Permutation Entropy (order=3)
    features["entropy_perm"] = ant.perm_entropy(steer, order=3, normalize=True)

    # 3) High Frequency Energy (0.3–1 Hz)

    band_energy, total_energy = calcBandpower(steer, fs, 0.3, 1.0)

    features["hf_energy_abs"] = band_energy
    features["hf_energy_ratio"] = band_energy / total_energy if total_energy > 0 else 0

    # 4) Micro-Correction Features (Δsteer)

    dsteer = np.diff(steer)

    # sign changes
    sign_changes = np.sum(np.diff(np.sign(dsteer)) != 0)
    features["micro_sign_changes"] = sign_changes

    # distances between sign changes
    change_idx = np.where(np.diff(np.sign(dsteer)) != 0)[0]

    if len(change_idx) > 1:
        intervals = np.diff(change_idx)
        features["micro_typical_interval"] = np.median(intervals)
        features["micro_mean_interval_time"] = np.mean(intervals) * dt
    else:
        features["micro_typical_interval"] = 0
        features["micro_mean_interval_time"] = 0

    # 5) Steering Path Length

    features["steer_path_length"] = np.sum(np.abs(dsteer))

    # 6) Cross Features with Lane Offset (optional)

    if lane_col is not None and lane_col in window_df.columns:

        lane = window_df[lane_col].values

        # Cross correlation max
        corr = np.correlate(
            steer - np.mean(steer),
            lane - np.mean(lane),
            mode="full"
        )
        features["crosscorr_max"] = np.max(np.abs(corr))

        # Efficiency: steering per lane RMS
        lane_rms = np.sqrt(np.mean(lane**2))
        features["correction_efficiency"] = (
            features["steer_path_length"] / lane_rms
            if lane_rms > 0 else 0
        )

    return features


In [6]:
# Liest CSVs ein und gibt DataFrame zurück
def readData(location : str) -> pd.DataFrame:
    return pd.read_csv(location, header=0)

# Vorbereiten der Daten
def prepareDriverData(dataframe : pd.DataFrame, driverId : str) -> pd.DataFrame:
    # Umwandeln der Daten in numerische Werte, Löschen der ersten beiden Zeilen (weil diese Text bzw. immer 0 / NaN sind), Interpolieren von NaN werten, Fallenlassen von Spalten mit persistenten NaN-Werten
    dataframe = dataframe.apply(pd.to_numeric, errors="coerce").drop([0,1]).interpolate().dropna(axis=1, how="all")
    # Entfernen von Spalten mit konstanten Werten (weil diese sich nicht zum Unterscheiden von Fahrern eignen)
    dataframe = dataframe.loc[:, (dataframe != dataframe.iloc[0]).any()]
    dataframe["driver_id"] = driverId
    return dataframe

# Segmentieren des Datensatzes in Fenster
def segmentData(data : pd.DataFrame, windowSize : int) -> pd.DataFrame:
    
    if windowSize <= 0:
        raise ValueError("windowSize muss > 0 sein")

    # Sortieren nach Zeitstempel und zurücksetzen (dh. neu nummerieren) der automatisch erstellten Indizes entlang der neuen Sortierung
    data = data.sort_values("timestamp").reset_index(drop=True)
    data["window_id"] = -1  # Setzen eines Default-Werts, damit nicht standardmäßig 0 verwendet wird, weil die Nummerierung bei 0 startet

    windowId = 0
    #Diese Schleife iteriert von 0 bis zum Ende des Datensatzes mit einer Schrittweite von <windowSize>
    for start in range(0, len(data), windowSize):   
        end = min(start + windowSize, len(data))            # Vermeiden von Zugriffsfehlern, durch wählen des niedrigeren Indizes (Ende des Datensatzes bzw. reguläres Ende des Fensters)
        data.loc[start:end - 1, "window_id"] = windowId     # Einfügen der Fenster-ID in jeder Zeile des Fensters
        windowId += 1

    return data

# Einlesen, Vorbereiten und Segmentieren einer Liste Datensätzen (CSV-Dateien); Gibt Liste von DataFrames zurück
def readPrepareAndSegmentData(recordings : list, windowSize : int) -> list[pd.DataFrame]:
    dataframes = []
    for recording in recordings:
        dataframes.append(segmentData(prepareDriverData(readData(recording["path"]), recording["id"]), windowSize))
    return dataframes

# Zusammenführen von DataFrames zu einem einzelnen DataFrame; Gibt einzelnen DataFrame zurück
def joinAndReduceData(dataframes : list[pd.DataFrame]) -> pd.DataFrame:
    jointData = pd.concat(dataframes, ignore_index=True).fillna(0)  # Zusammenführen, dabei sicherheitshalber noch NaNs mit 0 ersetzen (kann ggf. weggelassen werden)
    print(jointData[["window_id","driver_id"]])

    # Auswählen aller Reifenrotationsgeschwindigkeiten, damit diese zusammengefasst werden können
    rotationVelocities = jointData[["car0_wheel0_rot_vel", "car0_wheel1_rot_vel", "car0_wheel2_rot_vel", "car0_wheel3_rot_vel"]]
    # Fallenlassen redundanter bzw. mit anderen Features stark korrelierender Eigenschaften, sowie der Reifenrotationsgeschwindigkeiten
    jointData = jointData.drop(columns=["car0_rpm", "car0_engine_rpm", "car0_velocity_vehicle", "car0_wheel0_rot_vel", "car0_wheel1_rot_vel", "car0_wheel2_rot_vel", "car0_wheel3_rot_vel"])
    # Einfügen des Durchschnitts der Reifenrotationsgeschwindigkeiten
    jointData["car0_wheel_avg_rot_vel"] = rotationVelocities.mean(axis=1)
    return jointData

In [7]:
def calculateSteeringFeaturesForWindowedData(data):
    all_features = []
    
    for window_id, win in data.groupby("window_id"):
        feats = calcSteeringFeatures(win)
        feats["window_id"] = window_id
        feats["driver_id"] = win["driver_id"].iloc[0]
        all_features.append(feats)

    feature_df = pd.DataFrame(all_features)
    return feature_df


In [19]:
def buildSteeringFeatureDatasetForFolder(
    folder="./recordings",
    window_size=50,
    steer_col="wheel_position",
    time_col="timestamp"
):
    """
    Reads all simulator recordings from 2026,
    extracts steering-based features per window,
    returns one big DataFrame:

    driver_id | run_id | window_id | jerk_mean | entropy_sample | ...
    """

    # ------------------------------------------------------------
    # 1) Glob all matching files from 2026
    # ------------------------------------------------------------
    pattern = os.path.join(folder, "recording_2026_*_*.csv")
    files = sorted(glob.glob(pattern))

    if len(files) == 0:
        raise FileNotFoundError("Keine 2026-Dateien gefunden!")

    print(f"Gefundene Dateien aus 2026: {len(files)}")

    # ------------------------------------------------------------
    # 2) Driver Mapping
    # ------------------------------------------------------------
    driver_map = {}
    driver_counter = 0

    all_rows = []

    # ------------------------------------------------------------
    # 3) Loop through each recording file (run)
    # ------------------------------------------------------------
    for run_id, filepath in enumerate(files):

        filename = os.path.basename(filepath)

        # Beispiel:
        # recording_2026_02_10__13_18_02_florian.csv
        driver_name = filename.split("_")[8].replace(".csv", "")

        # assign stable driver_id
        if driver_name not in driver_map:
            driver_map[driver_name] = driver_counter
            driver_counter += 1

        driver_id = driver_map[driver_name]

        print(f"Run {run_id}: Fahrer={driver_name} (ID={driver_id})")

        # --------------------------------------------------------
        # Load CSV
        # --------------------------------------------------------
        df = readData(filepath)

        # Ensure numeric + drop NaNs
        df = prepareDriverData(df, driver_id)

        # --------------------------------------------------------
        # Segment into windows
        # --------------------------------------------------------
        df = df.sort_values(time_col).reset_index(drop=True)
        df["window_id"] = df.index // window_size

        # --------------------------------------------------------
        # Extract steering features per window
        # --------------------------------------------------------
        for window_id, win in df.groupby("window_id"):

            feats = calcSteeringFeatures(
                win,
                steer_col=steer_col,
                time_col=time_col
            )

            # Metadata hinzufügen
            feats["driver_id"] = driver_id
            feats["driver_name"] = driver_name
            feats["run_id"] = run_id
            feats["window_id"] = window_id

            all_rows.append(feats)

    # ------------------------------------------------------------
    # 4) Final DataFrame
    # ------------------------------------------------------------
    feature_df = pd.DataFrame(all_rows)

    print("\nDriver Mapping:", driver_map)
    print("Final Feature DataFrame Shape:", feature_df.shape)

    return feature_df


In [20]:
features = buildSteeringFeatureDatasetForFolder("./recordings", 50, "wheel_position", "timestamp")

Gefundene Dateien aus 2026: 21
Run 0: Fahrer=fabian (ID=0)


C:\Users\games\AppData\Local\Temp\ipykernel_34408\3999157513.py:3: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,18,21,25,29,33,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,69,70,71,72,77,78) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(location, header=0)
p:\Apps\Entwicklung\Software\Anaconda\envs\MSuT\Lib\site-packages\nolds\measures.py:829: RuntimeWarning: Zero vectors are within tolerance for emb_dim and emb_dim + 1. Consider raising the tolerance parameter to avoid NaN result.
  warnings.warn(
p:\Apps\Entwicklung\Software\Anaconda\envs\MSuT\Lib\site-packages\nolds\measures.py:829: RuntimeWarning: Zero vectors are within tolerance for emb_dim and emb_dim + 1. Consider raising the tolerance parameter to avoid NaN result.
  warnings.warn(
p:\Apps\Entwicklung\Software\Anaconda\envs\MSuT\Lib\site-packages\nolds\measures.py:829: RuntimeWarning: Zero vectors are within tolerance for emb_d

ValueError: Shape of array too small to calculate a numerical gradient, at least (edge_order + 1) elements are required.

In [ ]:
recordings = [
    { "path": "./recordings/recording_2025_12_11__12_46_32_fabian.csv", "id": "Fabian" },
    { "path": "./recordings/recording_2025_12_11__12_29_26_florian.csv", "id": "Florian" },
    { "path": "./recordings/recording_2025_12_11__12_38_11_matthias.csv", "id": "Matthias" }
]

allDriverFeatures = []

for recording in recordings: 
    dataframes = readPrepareAndSegmentData([recording], 50)
    data = joinAndReduceData(dataframes)
    features = calculateSteeringFeaturesForWindowedData(data)
    allDriverFeatures.append(features)

df = pd.concat(allDriverFeatures)

C:\Users\games\AppData\Local\Temp\ipykernel_34408\3999157513.py:3: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,18,21,25,29,33,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,69,70,71,72,73,78) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(location, header=0)


       window_id driver_id
0              0    Fabian
1              0    Fabian
2              0    Fabian
3              0    Fabian
4              0    Fabian
...          ...       ...
34176        683    Fabian
34177        683    Fabian
34178        683    Fabian
34179        683    Fabian
34180        683    Fabian

[34181 rows x 2 columns]


p:\Apps\Entwicklung\Software\Anaconda\envs\MSuT\Lib\site-packages\nolds\measures.py:829: RuntimeWarning: Zero vectors are within tolerance for emb_dim and emb_dim + 1. Consider raising the tolerance parameter to avoid NaN result.
  warnings.warn(
C:\Users\games\AppData\Local\Temp\ipykernel_34408\3999157513.py:3: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,15,16,17,18,19,24,27,31,35,39,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,75,76,77,78) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(location, header=0)


       window_id driver_id
0              0   Florian
1              0   Florian
2              0   Florian
3              0   Florian
4              0   Florian
...          ...       ...
48406        968   Florian
48407        968   Florian
48408        968   Florian
48409        968   Florian
48410        968   Florian

[48411 rows x 2 columns]


p:\Apps\Entwicklung\Software\Anaconda\envs\MSuT\Lib\site-packages\nolds\measures.py:829: RuntimeWarning: Zero vectors are within tolerance for emb_dim and emb_dim + 1. Consider raising the tolerance parameter to avoid NaN result.
  warnings.warn(
p:\Apps\Entwicklung\Software\Anaconda\envs\MSuT\Lib\site-packages\nolds\measures.py:829: RuntimeWarning: Zero vectors are within tolerance for emb_dim and emb_dim + 1. Consider raising the tolerance parameter to avoid NaN result.
  warnings.warn(
p:\Apps\Entwicklung\Software\Anaconda\envs\MSuT\Lib\site-packages\nolds\measures.py:829: RuntimeWarning: Zero vectors are within tolerance for emb_dim and emb_dim + 1. Consider raising the tolerance parameter to avoid NaN result.
  warnings.warn(
p:\Apps\Entwicklung\Software\Anaconda\envs\MSuT\Lib\site-packages\nolds\measures.py:829: RuntimeWarning: Zero vectors are within tolerance for emb_dim and emb_dim + 1. Consider raising the tolerance parameter to avoid NaN result.
  warnings.warn(
p:\Apps\Entw

       window_id driver_id
0              0  Matthias
1              0  Matthias
2              0  Matthias
3              0  Matthias
4              0  Matthias
...          ...       ...
45940        918  Matthias
45941        918  Matthias
45942        918  Matthias
45943        918  Matthias
45944        918  Matthias

[45945 rows x 2 columns]


p:\Apps\Entwicklung\Software\Anaconda\envs\MSuT\Lib\site-packages\nolds\measures.py:829: RuntimeWarning: Zero vectors are within tolerance for emb_dim and emb_dim + 1. Consider raising the tolerance parameter to avoid NaN result.
  warnings.warn(
p:\Apps\Entwicklung\Software\Anaconda\envs\MSuT\Lib\site-packages\nolds\measures.py:829: RuntimeWarning: Zero vectors are within tolerance for emb_dim and emb_dim + 1. Consider raising the tolerance parameter to avoid NaN result.
  warnings.warn(
p:\Apps\Entwicklung\Software\Anaconda\envs\MSuT\Lib\site-packages\nolds\measures.py:829: RuntimeWarning: Zero vectors are within tolerance for emb_dim and emb_dim + 1. Consider raising the tolerance parameter to avoid NaN result.
  warnings.warn(
p:\Apps\Entwicklung\Software\Anaconda\envs\MSuT\Lib\site-packages\nolds\measures.py:829: RuntimeWarning: Zero vectors are within tolerance for emb_dim and emb_dim + 1. Consider raising the tolerance parameter to avoid NaN result.
  warnings.warn(
p:\Apps\Entw

In [16]:
features.to_csv("./preprocessed/JointRecordingsAdvancedFeatures.csv")